# Tensor Operations

idris-ml tensors live in C (via the backend) and are accessed through FFI.
This notebook explores the low-level tensor API.

## Creating Scalars

In [1]:
:exec (let t = prim__createScalar 3.14 0 in
  putStrLn ("pi = " ++ show (prim__item t)))

pi = 3.14


The second argument to `prim__createScalar` is `requires_grad`:
- `0` = no gradient tracking
- `1` = track gradients (for learnable parameters)

In [2]:
:t prim__createScalar

Variable.prim__createScalar : Double -> Int -> AnyPtr


## Arithmetic

In [3]:
:exec (let a = prim__createScalar 2.0 1 in
  let b = prim__createScalar 3.0 0 in
  let c = prim__mul a b in
  putStrLn ("2 * 3 = " ++ show (prim__item c)))

2 * 3 = 6.0


In [4]:
:exec (let a = prim__createScalar 10.0 0 in
  let b = prim__createScalar 4.0 0 in
  let s = prim__sub a b in
  putStrLn ("10 - 4 = " ++ show (prim__item s)))

10 - 4 = 6.0


## Creating 1D Tensors

For multi-element tensors, allocate a buffer, fill it, then create.

In [5]:
:exec (let buf = prim__setDouble
    (prim__setDouble
      (prim__setDouble (prim__allocDoubles 3) 0 1.0)
    1 2.0)
  2 3.0 in
  let t = prim__createState1d 3 buf in
  let s = prim__sum t in
  putStrLn ("sum [1, 2, 3] = " ++ show (prim__item s)))

sum [1, 2, 3] = 6.0


## One-Hot Encoding

`prim__oneHot` creates a one-hot vector from label indices — used for classification targets.

In [6]:
:exec (let lblBuf = prim__setInt (prim__allocInts 1) 0 2 in
  let oh = prim__oneHot lblBuf 1 5 in
  putStrLn ("one-hot(2, 5) = [" ++
    show (prim__item1d oh 0) ++ ", " ++
    show (prim__item1d oh 1) ++ ", " ++
    show (prim__item1d oh 2) ++ ", " ++
    show (prim__item1d oh 3) ++ ", " ++
    show (prim__item1d oh 4) ++ "]"))

one-hot(2, 5) = [0.0, 0.0, 1.0, 0.0, 0.0]


## Softmax and Log-Softmax

In [7]:
:exec (let buf = prim__setDouble
    (prim__setDouble
      (prim__setDouble (prim__allocDoubles 3) 0 1.0)
    1 2.0)
  2 3.0 in
  let logits = prim__createState1d 3 buf in
  let probs = prim__softmax logits 0 in
  putStrLn ("softmax [1,2,3] = [" ++
    show (prim__item1d probs 0) ++ ", " ++
    show (prim__item1d probs 1) ++ ", " ++
    show (prim__item1d probs 2) ++ "]"))

softmax [1,2,3] = [0.09003057317038046, 0.24472847105479764, 0.6652409557748218]


## Backend Info

Check which backend is active.

In [8]:
:exec putStrLn ("Backend: " ++ backendName)

Backend: tape
